In [32]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score,f1_score
from tqdm.notebook import tqdm
import os
import anndata as ad
import matplotlib.pyplot as plt

In [25]:
from sklearn.ensemble import RandomForestClassifier
def random_forest_classifier(
      adata,
      y_col='geo_region_of_origin',
      n_estimators=500,
      max_features='sqrt',
      label='90/10',
      random_state=0,
  ):
    # encode classes
    classes = sorted(adata.obs[y_col].unique().tolist())
    class_dict = {c: i for i, c in enumerate(classes)}
    adata.obs['class_id'] = adata.obs[y_col].map(class_dict)


    # remove classes with too few samples
    fold_ids = sorted(list(adata.obsm.keys()))

    class_counts = adata.obs['class_id'].value_counts()
    keep_classes = class_counts[class_counts >= len(fold_ids)].index
    adata = adata[adata.obs['class_id'].isin(keep_classes)].copy()
    
    fold_ids = sorted(list(adata.obsm.keys()))
    adata.obs['fold_' + label] = None
    for i, fold in enumerate(fold_ids):
        fold_col = adata.obsm[fold][:, -1]
        test_mask = fold_col == 1
        train_mask = ~test_mask
        X_train = adata.obsm[fold][train_mask, :-1]
        X_test  = adata.obsm[fold][test_mask,  :-1]
        y_train = adata.obs.loc[train_mask, 'class_id'].to_numpy()

        clf = RandomForestClassifier(
              n_estimators=n_estimators,
              max_features=max_features,
              random_state=random_state,
              n_jobs=-1,
          )
        clf.fit(X_train, y_train)
        preds = clf.predict(X_test)

        adata.obs.loc[test_mask, 'pred'] = preds
        adata.obs.loc[test_mask, 'fold_' + label] = i

    return adata


In [35]:
n_tree_vals = [50]
split_files = sorted([f for f in os.listdir('../data/') if f.startswith('pcScores')])
f1s = np.zeros((len(split_files), len(n_tree_vals), 10))

for s, split_file in enumerate(split_files):
    adata = ad.read_h5ad(os.path.join('../data', split_file))
    split_no = split_file.split('.')[0][-2:]

    for j, n_trees in tqdm(list(enumerate(n_tree_vals)), desc=split_file):
        result = random_forest_classifier(adata.copy(), n_estimators=n_trees, label=split_no)
        print(result)
        fold_col = 'fold_' + split_no
        folds = result.obs[fold_col].dropna().unique()
        for f, fold in enumerate(folds):
            sub = result[result.obs[fold_col] == fold].obs
            print(s, j, f)
            f1s[s, j, f] = f1_score(sub['class_id'].to_numpy(),
                                      sub['pred'].to_numpy(),
                                      average='macro')

pcScores_split_10.h5ad:   0%|          | 0/1 [00:00<?, ?it/s]

AnnData object with n_obs × n_vars = 1107 × 2810
    obs: 'hgdp_id', 'population_id', 'population_name', 'country_of_origin', 'geo_region_of_origin', 'genotyping_id', 'sex', 'class_id', 'fold_10', 'pred'
    obsm: 'X_pca_fold_1', 'X_pca_fold_10', 'X_pca_fold_2', 'X_pca_fold_3', 'X_pca_fold_4', 'X_pca_fold_5', 'X_pca_fold_6', 'X_pca_fold_7', 'X_pca_fold_8', 'X_pca_fold_9'
0 0 0
0 0 1
0 0 2
0 0 3
0 0 4
0 0 5
0 0 6
0 0 7
0 0 8
0 0 9


pcScores_split_20.h5ad:   0%|          | 0/1 [00:00<?, ?it/s]

AnnData object with n_obs × n_vars = 1107 × 2810
    obs: 'hgdp_id', 'population_id', 'population_name', 'country_of_origin', 'geo_region_of_origin', 'genotyping_id', 'sex', 'class_id', 'fold_20', 'pred'
    obsm: 'X_pca_fold_1', 'X_pca_fold_2', 'X_pca_fold_3', 'X_pca_fold_4', 'X_pca_fold_5'
1 0 0
1 0 1
1 0 2
1 0 3
1 0 4


pcScores_split_25.h5ad:   0%|          | 0/1 [00:00<?, ?it/s]

AnnData object with n_obs × n_vars = 1107 × 2810
    obs: 'hgdp_id', 'population_id', 'population_name', 'country_of_origin', 'geo_region_of_origin', 'genotyping_id', 'sex', 'class_id', 'fold_25', 'pred'
    obsm: 'X_pca_fold_1', 'X_pca_fold_2', 'X_pca_fold_3', 'X_pca_fold_4'
2 0 0
2 0 1
2 0 2
2 0 3


pcScores_split_33.h5ad:   0%|          | 0/1 [00:00<?, ?it/s]

AnnData object with n_obs × n_vars = 1107 × 2810
    obs: 'hgdp_id', 'population_id', 'population_name', 'country_of_origin', 'geo_region_of_origin', 'genotyping_id', 'sex', 'class_id', 'fold_33', 'pred'
    obsm: 'X_pca_fold_1', 'X_pca_fold_2', 'X_pca_fold_3'
3 0 0
3 0 1
3 0 2


In [ ]:
splits = ['10', '20', '25', '33']
n_folds_per_split = [10, 5, 4, 3]

mean_scores = []
for s, n_folds in enumerate(n_folds_per_split):
    fold_scores = [f1s[s, 0, f] for f in range(n_folds)]
    mean_scores.append(np.mean(fold_scores))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(splits, mean_scores)
ax.set_xlabel('Test split %')
ax.set_ylabel('F1 score (macro avg)')
ax.set_title('Mean F1 score by PCA split')
ax.set_ylim(.5, 1)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

[[[0.83439131 0.78248787 0.80065037 0.82958477 0.80235684 0.81586566
   0.72295215 0.82455742 0.69048773 0.77195543]]

 [[0.79083389 0.68798477 0.8115712  0.80586505 0.76695077 0.
   0.         0.         0.         0.        ]]

 [[0.77773981 0.79416012 0.81720342 0.78813018 0.         0.
   0.         0.         0.         0.        ]]

 [[0.79518956 0.75993309 0.79594647 0.         0.         0.
   0.         0.         0.         0.        ]]]


In [ ]:
param_grid = {
    "max_depth":         [5, 10, 20, None],
    "max_features":      ["sqrt", "log2", 0.1, 0.25, 0.5],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf":  [1, 2, 5, 10],
    "bootstrap":         [True]
}

adata = ad.read_h5ad('../data/pcScores_split_20.h5ad')
fold_keys = [k for k in adata.obsm.keys() if k.startswith('X_pca_fold')]
print(f"=== pcScores_split_20.h5ad | {len(fold_keys)} folds ===")

all_results = []

for fold_key in fold_keys:
    fold_no = fold_key.split('_')[-1]
    X = adata.obsm[fold_key]
    Y = adata.obs['geo_region_of_origin']

    print(f"  Running grid search on fold {fold_no}...")

    RFO = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    grid_search = GridSearchCV(
        estimator=RFO,
        param_grid=param_grid,
        scoring='f1_macro',
        cv=5,
        n_jobs=-1,
        verbose=0
    )
    grid_search.fit(X, Y)

    for i, params in enumerate(grid_search.cv_results_['params']):
        all_results.append({
            'fold':              fold_no,
            'max_depth':         params['max_depth'],
            'max_features':      params['max_features'],
            'min_samples_split': params['min_samples_split'],
            'min_samples_leaf':  params['min_samples_leaf'],
            'bootstrap':         params['bootstrap'],
            'f1':                grid_search.cv_results_['mean_test_score'][i]
        })

    # ← saves after every fold in case of crash
    pd.DataFrame(all_results).to_csv('../data/gridsearch_results.csv', index=False)
    print(f"  Fold {fold_no} done — results saved")